In [ ]:
# Phase 4: land the clean, PII-free AA fleet reference table into lh_bronze.
#
# Unlike nb_bronze_bts, this notebook does NOT download anything -- the
# FAA Releasable Aircraft Database is bot-protected against automated
# requests (see docs/decisions/0001-bot-protected-sources-need-manual-export.md),
# and the join/filter/PII-drop (src/hubops/faa_registry.py) already ran
# locally, BEFORE anything with owner names/addresses touched the cloud --
# a stronger privacy posture than landing raw MASTER.txt and dropping
# columns in a later cell. This notebook just reads the already-clean
# result (uploaded via docs/decisions/0006-onelake-binary-upload-needs-powershell.md's
# method) and writes it as a Delta table.

from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import IntegerType

WORKSPACE_ID = "16b2a9c6-6418-485f-865f-c47442b8b6c6"
LH_BRONZE_ID = "09a8b1f5-f33c-4eed-912d-40b9cbdc9b06"
ABFSS_ROOT = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LH_BRONZE_ID}"

csv_path = f"{ABFSS_ROOT}/Files/faa/dim_aircraft_aa.csv"
df = spark.read.option("header", True).csv(csv_path)
print(f"read {df.count():,} rows from {csv_path}")


In [ ]:
for int_col in ["seats", "num_engines", "year_mfr"]:
    df = df.withColumn(int_col, col(int_col).cast(IntegerType()))

df = (
    df.withColumn("_source_file", lit("dim_aircraft_aa.csv"))
    .withColumn("_loaded_at", current_timestamp())
)
df.printSchema()


In [ ]:
# Full-snapshot overwrite is correct here (not replaceWhere like
# nb_bronze_bts): this is a reference table re-derived from the current
# FAA registry + current AA fleet, not an append-only fact table with
# meaningful history per period.
table_path = f"{ABFSS_ROOT}/Tables/dim_aircraft"
df.write.format("delta").mode("overwrite").save(table_path)
print(f"wrote {table_path}")


In [ ]:
check = spark.read.format("delta").load(table_path)
print(f"dim_aircraft row count: {check.count():,}")
check.groupBy("manufacturer", "model").count().orderBy(col("count").desc()).show(15, truncate=False)
